In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

grid   = pd.read_parquet('../data/processed/daily_sales.parquet')
sample = pd.read_csv('../data/raw/sample_submission.csv')
model  = joblib.load('../models/hgbr_model.pkl')

print(f'Grid shape     : {grid.shape}')
print(f'Sample rows    : {len(sample)}')
print(f'Model loaded   : ../models/hgbr_model.pkl')

Grid shape     : (1754, 15972)
Sample rows    : 31944
Model loaded   : ../models/hgbr_model.pkl


In [2]:
LAG_DAYS     = [1, 2, 3, 7, 14, 21, 28, 35, 42, 56]
ROLL_WINDOWS = [7, 14, 28, 56, 90]
DOW_LAG_WKS  = [1, 2, 4, 8, 12]

def build_features(d, series):
    hist = series[series.index < d]
    row  = {
        'dow'           : d.dayofweek,
        'month'         : d.month,
        'dom'           : d.day,
        'quarter'       : d.quarter,
        'week'          : int(d.isocalendar().week),
        'is_weekend'    : int(d.dayofweek >= 5),
        'is_tet'        : int((d.month==1 and d.day>=15) or
                              (d.month==2 and d.day<=15)),
        'is_month_end'  : int(d.day >= 25),
        'is_month_start': int(d.day <= 5),
    }
    for lag in LAG_DAYS:
        row[f'lag_{lag}'] = float(
            series.get(d - pd.Timedelta(days=lag), 0.0))

    for w in ROLL_WINDOWS:
        win = hist.iloc[-w:] if len(hist) >= w else hist
        row[f'rmean_{w}'] = float(win.mean())
        row[f'rstd_{w}']  = float(win.std()) if len(win) > 1 else 0.
        row[f'rmax_{w}']  = float(win.max())
        row[f'rpos_{w}']  = float((win > 0).mean())

    same_dow = hist[hist.index.dayofweek == d.dayofweek]
    for wk in DOW_LAG_WKS:
        row[f'dlag_{wk}w'] = float(same_dow.iloc[-wk]) \
                              if len(same_dow) >= wk else 0.

    t28 = hist.iloc[-28:]
    row['trend_28'] = float(
        np.polyfit(np.arange(len(t28)), t28.values.astype(float), 1)[0]
    ) if len(t28) > 2 else 0.

    row['zero_frac_28'] = float((t28 == 0).mean()) if len(t28) > 0 else 1.

    ly     = d - pd.DateOffset(years=1)
    ly_win = hist[(hist.index >= ly - pd.Timedelta(days=14)) &
                  (hist.index <= ly + pd.Timedelta(days=14))]
    row['ly_mean'] = float(ly_win.mean()) if len(ly_win) > 0 else 0.

    return row

print(f'build_features ready — 47 features')

build_features ready — 47 features


In [3]:
SKU_ORDER  = (sample[sample['id'].str.endswith('_validation')]
              ['id'].str.replace('_validation', '').tolist())
VAL_DATES  = pd.date_range('2025-09-06', '2025-10-03', freq='D')
EVAL_DATES = pd.date_range('2025-10-04', '2025-10-31', freq='D')
F_COLS     = [f'F{i}' for i in range(1, 29)]
FEAT_COLS  = list(build_features(pd.Timestamp('2025-01-01'),
                                  grid[grid.columns[0]]).keys())

print(f'SKUs in submission : {len(SKU_ORDER)}')
print(f'Val dates          : {VAL_DATES[0].date()} → {VAL_DATES[-1].date()}')
print(f'Eval dates         : {EVAL_DATES[0].date()} → {EVAL_DATES[-1].date()}')
print(f'Feature columns    : {len(FEAT_COLS)}')

SKUs in submission : 15972
Val dates          : 2025-09-06 → 2025-10-03
Eval dates         : 2025-10-04 → 2025-10-31
Feature columns    : 47


In [4]:
qty_90d  = grid[grid.index >= '2025-06-07'].sum()
qty_365d = grid[grid.index >= '2024-09-06'].sum()
qty_28d  = grid[grid.index >= '2025-08-09'].sum()

TIER_A = qty_90d[(qty_90d >= 150) & (qty_365d >= 400)].index.tolist()
TIER_B = qty_90d[(qty_90d >= 30)  & (qty_365d >= 80) &
                 (~qty_90d.index.isin(TIER_A))].index.tolist()
TIER_C = [sku for sku in SKU_ORDER
          if sku not in TIER_A and sku not in TIER_B]

print(f'Tier A — ML forecast     : {len(TIER_A):>5} SKUs')
print(f'Tier B — DOW avg         : {len(TIER_B):>5} SKUs')
print(f'Tier C — Zero forecast   : {len(TIER_C):>5} SKUs')

Tier A — ML forecast     :   128 SKUs
Tier B — DOW avg         :   376 SKUs
Tier C — Zero forecast   : 15468 SKUs


In [5]:
def dow_forecast(series, dates, n_weeks=12):
    s = series.copy()
    preds = []
    if series.iloc[-56:].sum() == 0:
        return np.zeros(len(dates))
    for d in dates:
        if d.dayofweek >= 5:
            preds.append(0.0)
            s[d] = 0.0
            continue
        past = s[s.index.dayofweek == d.dayofweek].iloc[-n_weeks:]
        if len(past) > 0 and past.sum() > 0:
            w    = np.exp(np.linspace(-2, 0, len(past)))
            pred = float(np.average(past.values, weights=w))
        else:
            pred = float(s.iloc[-28:].mean())
        pred = max(0.0, pred)
        preds.append(pred)
        s[d] = pred
    return np.array(preds)

print('dow_forecast ready')

dow_forecast ready


In [6]:
forecast_cache = {}   # sku → (val_28, eval_28)

print(f'Forecasting {len(TIER_B)} Tier B SKUs...')
for i, sku in enumerate(TIER_B):
    val_pred  = dow_forecast(grid[sku], VAL_DATES)
    s_ext     = pd.concat([grid[sku],
                           pd.Series(val_pred, index=VAL_DATES)])
    eval_pred = dow_forecast(s_ext, EVAL_DATES)
    forecast_cache[sku] = (val_pred, eval_pred)

    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(TIER_B)} done...')

print(f'Tier B done ✓')

Forecasting 376 Tier B SKUs...
  100/376 done...
  200/376 done...
  300/376 done...
Tier B done ✓


In [7]:
print(f'Forecasting {len(TIER_A)} Tier A SKUs with ML...')

for i, sku in enumerate(TIER_A):
    series = grid[sku].copy()

    # Validation window
    val_preds = []
    for d in VAL_DATES:
        row  = build_features(d, series[series.index < d])
        Xrow = pd.DataFrame([row], columns=FEAT_COLS).fillna(0)
        pred = max(0.0, float(np.expm1(model.predict(Xrow)[0])))
        val_preds.append(pred)
        series[d] = pred

    # Evaluation window
    eval_preds = []
    for d in EVAL_DATES:
        row  = build_features(d, series[series.index < d])
        Xrow = pd.DataFrame([row], columns=FEAT_COLS).fillna(0)
        pred = max(0.0, float(np.expm1(model.predict(Xrow)[0])))
        eval_preds.append(pred)
        series[d] = pred

    forecast_cache[sku] = (np.array(val_preds), np.array(eval_preds))

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(TIER_A)} done...')

print(f'Tier A done ✓')

Forecasting 128 Tier A SKUs with ML...
  20/128 done...
  40/128 done...
  60/128 done...
  80/128 done...
  100/128 done...
  120/128 done...
Tier A done ✓


In [8]:
ZEROS = np.zeros(28)

rows = []

# Validation block
for sku in SKU_ORDER:
    vp, _ = forecast_cache.get(sku, (ZEROS, ZEROS))
    vp    = np.maximum(vp, 0.0)
    rows.append([f'{sku}_validation'] + vp.tolist())

# Evaluation block
for sku in SKU_ORDER:
    _, ep = forecast_cache.get(sku, (ZEROS, ZEROS))
    ep    = np.maximum(ep, 0.0)
    rows.append([f'{sku}_evaluation'] + ep.tolist())

submission = pd.DataFrame(rows, columns=['id'] + F_COLS)
print(f'Submission shape : {submission.shape}')

Submission shape : (31944, 29)


In [9]:
assert list(submission['id']) == list(sample['id']), 'ID order mismatch!'
assert len(submission) == 31944,                      'Wrong row count!'
assert submission['id'].nunique() == 31944,           'Duplicate IDs!'
assert (submission[F_COLS].values >= 0).all(),        'Negatives found!'

vals         = submission[F_COLS].values.flatten()
nonzero_rows = (submission[F_COLS].sum(axis=1) > 0).sum()

print('=' * 40)
print('SUBMISSION SUMMARY')
print('=' * 40)
print(f'Rows             : {len(submission):,}  ✓')
print(f'ID order         : matches sample  ✓')
print(f'Negatives        : 0  ✓')
print(f'Duplicate IDs    : 0  ✓')
print(f'Non-zero rows    : {nonzero_rows:,} / {len(submission):,}')
print(f'% zero cells     : {(vals==0).mean():.1%}')
print(f'Mean forecast    : {vals.mean():.4f}')
print(f'Max forecast     : {vals.max():.2f}')
print(f'Total units pred : {vals.sum():,.0f}')

SUBMISSION SUMMARY
Rows             : 31,944  ✓
ID order         : matches sample  ✓
Negatives        : 0  ✓
Duplicate IDs    : 0  ✓
Non-zero rows    : 978 / 31,944
% zero cells     : 97.7%
Mean forecast    : 0.0292
Max forecast     : 50.64
Total units pred : 26,111


In [10]:
submission.to_csv('../data/submission/submission_final.csv', index=False)
print('✅ Saved → ../data/submission/submission_final.csv')
print()
print('Sample rows:')
print(submission.head(3)[['id','F1','F2','F3','F4','F5']].to_string(index=False))
print('...')
print(submission.iloc[15972:15975][['id','F1','F2','F3','F4','F5']].to_string(index=False))

✅ Saved → ../data/submission/submission_final.csv

Sample rows:
                  id       F1  F2       F3       F4       F5
SKU-00001_validation 0.000000 0.0 0.000000 0.000000 0.000000
SKU-00002_validation 1.580871 0.0 3.471552 3.674520 3.136323
SKU-00003_validation 3.833805 0.0 5.625149 7.725626 5.672609
...
                  id       F1  F2       F3       F4       F5
SKU-00001_evaluation 0.000000 0.0 0.000000 0.000000 0.000000
SKU-00002_evaluation 2.545048 0.0 3.763872 3.211053 3.391074
SKU-00003_evaluation 4.088903 0.0 6.986891 5.914518 5.834534
